# 00 — Course setup and dataset

**Estimated time:** 60–90 minutes
**Prerequisites:** Python, pandas, scikit-learn basics, and introductory statistics.
**Depends on:** nothing.

## Learning objectives

- Frame the prediction problem at a precise decision moment.
- Load the course dataset from local files, not the network.
- Distinguish between a data catalog and a data dictionary.
- Spot leakage before modeling starts.
- Build a simple leakage-safe baseline with scikit-learn.
- Define development, validation, and sealed test splits.

The question for the course is: **before a call is placed, which clients should be
prioritized for a term-deposit marketing campaign?** This is a ranking-and-selection
problem, not a causal-effect problem.

**Teaching note:** students often jump straight to model tuning. Here we slow down and
treat dataset choice, prediction moment, and split design as first-class modeling steps.


In [11]:
import hashlib, json, os, platform, random, warnings
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
# Known interoperability/UI warnings do not affect predictions or notebook execution.
warnings.filterwarnings("ignore", message="X does not have valid feature names, but LGBMClassifier")
warnings.filterwarnings("ignore", message="IProgress not found.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (balanced_accuracy_score, brier_score_loss, confusion_matrix,
                             f1_score, log_loss, precision_score, recall_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
TARGET = "y"
LEAKAGE_COLUMNS = ["duration"]

def project_root():
    '''Return the course root when present, otherwise the notebook directory.'''
    # Return the course root when present, otherwise the notebook's directory.
    return ROOT

def set_seed(seed=SEED):
    '''Seed Python and NumPy RNGs for reproducible notebook runs.'''
    random.seed(seed)
    np.random.seed(seed)

def fast_mode():
    '''Report whether notebooks should use the reduced fast-mode sample.'''
    # Set FAST_MODE=0 for full-size experiments; laptop mode is the default.
    return os.getenv("FAST_MODE", "1").lower() not in {"0", "false", "no"}

def bank_data_path():
    '''Locate the bundled Bank Marketing CSV file.'''
    # The course ships with a local dataset; notebooks never access the network.
    path = project_root() / "data" / "raw" / "bank-full.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"Expected the bundled Bank Marketing data at {path}. "
            "Run the notebook from the course root or place bank-full.csv there."
        )
    return path

def file_sha256(path):
    '''Compute the SHA-256 digest of a local file.'''
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def load_bank_data(include_duration=False):
    '''Load the Bank Marketing dataset and drop leakage columns by default.'''
    # Load the data, encode y, and exclude post-call duration by default.
    frame = pd.read_csv(bank_data_path(), sep=";")
    frame[TARGET] = frame[TARGET].map({"no": 0, "yes": 1}).astype("int8")
    if not include_duration:
        frame = frame.drop(columns=LEAKAGE_COLUMNS)
    return frame

def stratified_sample(frame, n, seed=SEED):
    '''Draw a label-preserving sample from a classified dataset.'''
    if n >= len(frame):
        return frame.copy()
    fractions = frame[TARGET].value_counts(normalize=True)
    counts = (fractions * n).round().astype(int)
    counts.iloc[0] += n - counts.sum()
    parts = [group.sample(n=min(counts.loc[label], len(group)),
                          random_state=seed + int(label))
             for label, group in frame.groupby(TARGET)]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

def make_splits(frame=None, reduced=None):
    '''Create deterministic stratified train, validation, and test splits.'''
    # Deterministic stratified 60/20/20 split; test stays sealed until notebook 09.
    from sklearn.model_selection import train_test_split
    frame = load_bank_data() if frame is None else frame
    train_val, test = train_test_split(
        frame, test_size=0.20, stratify=frame[TARGET], random_state=SEED)
    train, validation = train_test_split(
        train_val, test_size=0.25, stratify=train_val[TARGET], random_state=SEED)
    reduced = fast_mode() if reduced is None else reduced
    if reduced:
        train = stratified_sample(train, 12_000)
        validation = stratified_sample(validation, 4_000, SEED + 1)
        test = stratified_sample(test, 4_000, SEED + 2)
    return tuple(part.reset_index(drop=True) for part in (train, validation, test))

def split_xy(frame):
    '''Separate a frame into feature matrix and target vector.'''
    return frame.drop(columns=TARGET), frame[TARGET]

def feature_groups(frame):
    '''Identify numeric and categorical feature columns.'''
    features = frame.drop(columns=[TARGET], errors="ignore")
    categorical = features.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numerical = features.select_dtypes(include=np.number).columns.tolist()
    return numerical, categorical

def make_preprocessor(frame, scale_numeric=True):
    '''Build the preprocessing pipeline for numeric and categorical features.'''
    # Preprocessing is fitted only inside the enclosing training/CV pipeline.
    numerical, categorical = feature_groups(frame)
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=10, sparse_output=True)),
    ])
    return ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numerical),
        ("categorical", categorical_pipe, categorical),
    ], sparse_threshold=0.3)

def classification_metrics(y_true, probability, threshold=0.5):
    '''Compute ranking and threshold-based classification metrics.'''
    prediction = np.asarray(probability) >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {"log_loss": log_loss(y_true, probability),
            "brier_score": brier_score_loss(y_true, probability),
            "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
            "f1": f1_score(y_true, prediction, zero_division=0),
            "precision": precision_score(y_true, prediction, zero_division=0),
            "recall": recall_score(y_true, prediction, zero_division=0),
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "cost": float(fp + 5 * fn)}

def threshold_table(y_true, probability, thresholds=None):
    '''Evaluate classification metrics across a list of decision thresholds.'''
    thresholds = np.linspace(0.05, 0.80, 76) if thresholds is None else thresholds
    return pd.DataFrame([{"threshold": float(t),
                          **classification_metrics(y_true, probability, float(t))}
                         for t in thresholds])

def add_domain_features(frame):
    '''Create domain-inspired features for the Bank Marketing dataset.'''
    result = frame.copy()
    result["was_previously_contacted"] = (result["pdays"] != -1).astype("int8")
    result["pdays_clean"] = result["pdays"].replace(-1, np.nan)
    result["contact_pressure"] = result["campaign"] / (1 + result["previous"])
    result["balance_per_age"] = result["balance"] / result["age"].clip(lower=18)
    result["age_band"] = pd.cut(result["age"], bins=[0, 29, 39, 49, 59, np.inf],
                                labels=["<30", "30s", "40s", "50s", "60+"]).astype("object")
    return result.drop(columns=["pdays"])

def environment_metadata():
    '''Collect runtime metadata for experiment tracking.'''
    import sklearn
    return {"python": platform.python_version(), "platform": platform.platform(),
            "numpy": np.__version__, "pandas": pd.__version__,
            "scikit_learn": sklearn.__version__}

def write_json(data, path):
    '''Serialize structured data to a JSON file on disk.'''
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")

set_seed(SEED)
FAST_MODE = fast_mode()
CV_FOLDS = 3 if FAST_MODE else 5
BRAND_COLOR = "#2a9d8f"
ACCENT_COLOR = "#e76f51"
sns.set_theme(style="whitegrid", context="notebook", palette=[BRAND_COLOR, ACCENT_COLOR, "#264653", "#f4a261"])
plt.rcParams.update({
    "axes.facecolor": "#fcfcfc",
    "figure.facecolor": "white",
    "axes.edgecolor": "#d9d9d9",
    "grid.color": "#e8e8e8",
    "grid.linewidth": 0.8,
    "axes.titleweight": "bold",
    "axes.labelweight": "medium",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
pd.set_option("display.max_columns", 30)
print({"FAST_MODE": FAST_MODE, "CV_FOLDS": CV_FOLDS, "seed": SEED})


{'FAST_MODE': True, 'CV_FOLDS': 3, 'seed': 42}


## Why this dataset

We want a dataset that supports practical teaching:

- a realistic binary target,
- mixed numeric and categorical fields,
- clear leakage risks,
- enough size to show validation patterns,
- and a business context students can explain.

| Dataset | Why it works | Main drawback |
|---|---|---|
| UCI Bank Marketing | realistic campaign data, class imbalance, and obvious leakage risk | `duration` is post-call information |
| UCI Adult | useful for preprocessing practice | less aligned with a campaign decision workflow |
| UCI Online Shoppers | easy to understand conversion task | less instructive for campaign history and governance |

**Recommendation:** use Bank Marketing. It gives us a clean path through cataloging,
leakage checks, preprocessing, baseline modeling, and operational judgment.


In [12]:
path = bank_data_path()
raw = load_bank_data(include_duration=True)
print("cached path:", path)
print("shape:", raw.shape)
raw.head()


cached path: /Users/soroush/Desktop/Personal/Projects/Daneshkar/advanced_machine_learning/data/raw/bank-full.csv
shape: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,0
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,0
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,0
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,0
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,0


## Data catalog: making the dataset findable and governable

A **data catalog** records the context needed to discover, understand, trust, and govern
a data asset. A **data dictionary** is narrower: it explains individual fields.

In class, I like to treat the catalog as the answer to three questions:

1. What is this data for?
2. When is it safe to use?
3. What should stop us from using it?

![A raw dataset becomes a governed data asset through catalog metadata, enabling discovery, leakage-safe machine learning, and governance.](../assets/data_catalog_learning_flow.png)

*Teaching note:* the catalog is not just documentation. It is part of the modeling contract.


In [13]:
asset_catalog = pd.Series({
    "asset_name": "uci_bank_marketing",
    "local_path": str(path.relative_to(ROOT)),
    "source_url": "https://archive.ics.uci.edu/dataset/222/bank+marketing",
    "sha256": file_sha256(path),
    "rows": len(raw),
    "columns": raw.shape[1],
    "grain": "one row per marketing contact",
    "target": TARGET,
    "prediction_moment": "immediately before a scheduled call",
    "refresh_policy": "static course snapshot",
    "owner": "course maintainer",
    "steward": "simulated marketing-data steward",
}, name="value").to_frame()
display(asset_catalog)


,value
asset_name,uci_bank_marketing
local_path,data/raw/bank-full.csv
source_url,https://archive.ics.uci.edu/dataset/222/bank+m...
sha256,d1513ec63b385506f7cfce9f2c5caa9fe99e7ba4e8c3fa...
rows,45211
columns,17
grain,one row per marketing contact
target,y
prediction_moment,immediately before a scheduled call
refresh_policy,static course snapshot


## Field catalog and data dictionary

UCI documents `unknown` as an actual category; it is not silently converted to missing.
`pdays == -1` is a sentinel meaning the client was not previously contacted. `y=1`
means the client subscribed. Several fields describe the current campaign; availability
must be checked against the prediction moment rather than inferred from the column name.

The field catalog adds two useful governance properties:

- the field's role in the process,
- and whether the field is available at prediction time.

That makes the table behave like a feature contract, not just a schema dump.


In [14]:
field_roles = {
    "age": "client attribute", "job": "client attribute", "marital": "client attribute",
    "education": "client attribute", "default": "financial attribute",
    "balance": "financial attribute", "housing": "financial attribute", "loan": "financial attribute",
    "contact": "campaign context", "day": "campaign context", "month": "campaign context",
    "duration": "post-contact measurement", "campaign": "campaign history",
    "pdays": "campaign history", "previous": "campaign history", "poutcome": "campaign history",
    TARGET: "target",
}
schema = pd.DataFrame({
    "role": [field_roles[c] for c in raw.columns],
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "unique": raw.nunique(),
    "available_at_prediction": [c != "duration" and c != TARGET for c in raw.columns],
    "example": [raw[c].iloc[0] for c in raw.columns],
})
display(schema)
print("target distribution:")
display(raw[TARGET].value_counts().rename("count").to_frame().assign(rate=lambda x: x["count"] / len(raw)))


,role,dtype,missing,unique,available_at_prediction,example
age,client attribute,int64,0,77,True,58
job,client attribute,object,0,12,True,management
marital,client attribute,object,0,3,True,married
education,client attribute,object,0,4,True,tertiary
default,financial attribute,object,0,2,True,no
balance,financial attribute,int64,0,7168,True,2143
housing,financial attribute,object,0,2,True,yes
loan,financial attribute,object,0,2,True,no
contact,campaign context,object,0,3,True,unknown
day,campaign context,int64,0,31,True,5


target distribution:


,count,rate
y,,
0,39922,0.883015
1,5289,0.116985


In [15]:
categorical = raw.select_dtypes(include="object").columns.tolist()
numeric = raw.select_dtypes(include=np.number).columns.drop(TARGET).tolist()
print("categorical:", categorical)
print("numeric:", numeric)
display(raw[categorical].nunique().sort_values().rename("cardinality").to_frame())


categorical: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
numeric: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']


,cardinality
default,2
housing,2
loan,2
marital,3
contact,3
education,4
poutcome,4
job,12
month,12


### Minimum quality contract

Catalog descriptions are promises, not evidence. A useful catalog turns each important
promise into an explicit check, so we can tell the difference between a trustworthy
snapshot and a dataset that only looks clean at first glance.

For this snapshot, the minimum contract is intentionally small and strict:

- `y` must be binary and non-null, because the target is the label we train and evaluate
  against.
- `age` must stay in a plausible adult range, so obvious data-entry errors surface early.
- `day` and `month` must form valid calendar fields, because downstream feature logic
  assumes a real call date.
- `pdays` must obey the documented sentinel rule: `-1` means the client was not
  previously contacted, while non-negative values represent elapsed days since the
  last campaign contact.

These checks are not cosmetic. They are the boundary between a dataset we can trust and
one we should quarantine before it reaches modeling or reporting.

**Warning:** if a critical rule breaks in production data, the correct response is to fail
ingestion or quarantine the batch. Do not silently weaken the rule to make a pipeline pass.


In [16]:
quality_checks = pd.Series({
    "target is binary and non-null": raw[TARGET].notna().all() and set(raw[TARGET].unique()) <= {0, 1},
    "age is between 18 and 100": raw["age"].between(18, 100).all(),
    "day is between 1 and 31": raw["day"].between(1, 31).all(),
    "month uses documented abbreviations": set(raw["month"]) <= set("jan feb mar apr may jun jul aug sep oct nov dec".split()),
    "pdays is -1 or non-negative": ((raw["pdays"] == -1) | (raw["pdays"] >= 0)).all(),
    "source row count is stable": len(raw) == 45_211,
}, name="passed").to_frame()
display(quality_checks)
assert quality_checks["passed"].all(), "Critical catalog quality contract failed"


,passed
target is binary and non-null,True
age is between 18 and 100,True
day is between 1 and 31,True
month uses documented abbreviations,True
pdays is -1 or non-negative,True
source row count is stable,True


## Leakage audit: why `duration` must go

`duration` is only known during or after the call, so a pre-contact system cannot use it.
This is a classic leakage trap: the feature looks highly predictive, but only because it
contains information from the event we are trying to predict.

The next cell compares a simple logistic-regression pipeline with and without `duration`.
If the score jumps a lot, that is a warning sign, not a deployment win.


In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

audit = stratified_sample(raw, 12_000 if FAST_MODE else len(raw))
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

def leakage_audit(frame):
    X, y = frame.drop(columns=TARGET), frame[TARGET]
    model = Pipeline([
        ("preprocess", make_preprocessor(frame)),
        ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
    ])
    result = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=["neg_log_loss", "balanced_accuracy"],
        n_jobs=-1,
    )
    return {
        "log_loss (lower is better)": -result["test_neg_log_loss"].mean(),
        "balanced_accuracy (higher is better)": result["test_balanced_accuracy"].mean(),
    }

leakage_comparison = pd.DataFrame({
    "with_duration (invalid)": leakage_audit(audit),
    "without_duration (deployable)": leakage_audit(audit.drop(columns="duration")),
}).T
leakage_comparison


,log_loss (lower is better),balanced_accuracy (higher is better)
with_duration (invalid),0.244025,0.664468
without_duration (deployable),0.304622,0.582498


**Interpretation:** `duration` usually creates a dramatic apparent gain because successful
calls tend to last longer.

Other risks are subtler:

- `campaign` includes the current contact,
- `day` and `month` identify the current contact date,
- and `pdays`, `previous`, `poutcome` describe prior contact history.

We keep those fields only under the explicit assumption that scoring happens immediately
before a scheduled call. Change the decision moment and the feature contract must change too.


## The split contract

- **Development (60%)**: fitting, cross-validation, feature/model/hyperparameter selection.
- **Validation (20%)**: operating-threshold selection and limited finalist comparison.
- **Test (20%)**: sealed until notebook 09; used once for final evaluation.

A true temporal split would be preferable for deployment forecasting, but this file lacks a
year and stable full timestamp. A grouped split would help with repeated clients, but no
customer identifier is supplied.

**Best-practice warning:** do not use validation or test data to choose features, thresholds,
preprocessing choices, or the model family. That is how notebooks accidentally become less
trustworthy than they look.


In [18]:
clean = load_bank_data()  # duration excluded by default
development, validation, test = make_splits(clean, reduced=FAST_MODE)
split_summary = pd.DataFrame({
    "rows": [len(development), len(validation), len(test)],
    "positive_rate": [x[TARGET].mean() for x in (development, validation, test)],
}, index=["development", "validation", "test (sealed)"])
display(split_summary)
assert "duration" not in clean.columns
assert sum(len(x) for x in make_splits(clean, reduced=False)) == len(clean)


,rows,positive_rate
development,12000,0.117
validation,4000,0.117
test (sealed),4000,0.117


## Practical baseline

A strong course notebook should not stop at data inspection. We also want a small
deployable baseline that students can understand, modify, and compare against later
models.

The baseline below uses:

- the leakage-safe columns only,
- a standard preprocessing pipeline,
- and logistic regression as a conventional first model.

This is intentionally simple. If a sophisticated method cannot beat this setup, we should
be cautious about adding complexity.


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, brier_score_loss, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline

def split_xy(frame):
    return frame.drop(columns=TARGET), frame[TARGET]

baseline = Pipeline([
    ("preprocess", make_preprocessor(development)),
    ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
])

X_dev, y_dev = split_xy(development)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
dev_prob = cross_val_predict(baseline, X_dev, y_dev, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

baseline_summary = pd.Series({
    "log_loss": log_loss(y_dev, dev_prob),
    "brier_score": brier_score_loss(y_dev, dev_prob),
    "balanced_accuracy@0.5": balanced_accuracy_score(y_dev, dev_prob >= 0.5),
    "f1@0.5": f1_score(y_dev, dev_prob >= 0.5, zero_division=0),
}, name="development CV")
display(baseline_summary.to_frame())


,development CV
log_loss,0.307628
brier_score,0.088289
balanced_accuracy@0.5,0.562812
f1@0.5,0.224148


## Evaluation and interpretation

For imbalanced problems, accuracy alone is usually misleading. We prefer:

- log loss and Brier score for probability quality,
- precision and recall for positive-class usefulness,
- balanced accuracy for a simple class-balance check,
- and cost for the operational trade-off we actually care about.

In [21]:
dev_pred = (dev_prob >= 0.5).astype(int)
interpretation = pd.DataFrame({
    "predicted_positive_rate": [dev_pred.mean()],
    "actual_positive_rate": [y_dev.mean()],
    "precision": [precision_score(y_dev, dev_pred, zero_division=0)],
    "recall": [recall_score(y_dev, dev_pred, zero_division=0)],
}, index=["development CV"])
display(interpretation)

,predicted_positive_rate,actual_positive_rate,precision,recall
development CV,0.02725,0.117,0.593272,0.138177


## How to interpret this table

- `predicted_positive_rate` is the share of samples the model labeled as positive.
- `actual_positive_rate` is the true positive-class prevalence in the development set.
- If the predicted rate is much lower than the actual rate, the model is being conservative and missing many positives.
- `precision` answers: among the samples predicted positive, how many were actually positive?
- `recall` answers: among all actual positives, how many did the model find?

### Takeaway

A model with decent precision but low recall is selective: its positive predictions are fairly trustworthy, but it misses most real positives. On an imbalanced problem like this, that usually means the default `0.5` threshold is too strict and should be tuned based on the business goal.


## Practical recommendation

Use Bank Marketing as the course dataset, but keep the prediction contract strict:

- exclude `duration`,
- document when each field is available,
- use pipelines so preprocessing stays inside cross-validation,
- and keep the test set sealed until the final evaluation notebook.

If we later improve the model, the main gain should come from better feature design,
threshold selection, and validation discipline, not from hiding leakage.


## Exercises

1. Rewrite the asset catalog for a production setting with a named owner and freshness SLA.
2. Add one cross-field rule involving `pdays`, `previous`, and `poutcome`.
3. Change the prediction moment to "one week before the campaign" and identify which fields
   become unavailable.
4. Replace logistic regression with a dummy classifier and explain why it is still a useful baseline.
5. Propose one evaluation metric you would show a marketing stakeholder and one you would show an ML engineer.

## Summary

We selected a dataset, documented its contract, checked basic quality rules, audited leakage,
and built a small leakage-safe baseline. That sets up the rest of the course with a common
language for features, splits, and evaluation.

## References

- [UCI Bank Marketing dataset](https://archive.ics.uci.edu/dataset/222/bank+marketing)
- [Dataset paper (Decision Support Systems)](https://doi.org/10.1016/j.dss.2014.03.001)
- [Data Catalog Vocabulary (DCAT) v3](https://www.w3.org/TR/vocab-dcat-3/)
- [scikit-learn model selection](https://scikit-learn.org/stable/model_selection.html)
